In [3]:
import os
import numpy as np
import shutil
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# Install necessary libraries
# !pip install evaluate accelerate
import evaluate

# 1. Load Dataset
dataset = load_dataset("ag_news")

# 2. Reduce Dataset Size (FAST TRAINING)
small_train = dataset["train"].shuffle(seed=42).select(range(2000))
small_test = dataset["test"].shuffle(seed=42).select(range(500))

dataset = {
    "train": small_train,
    "test": small_test
}

# 3. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# 4. Tokenization Function
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# Apply tokenization
tokenized_train = dataset["train"].map(tokenize_function, batched=True)
tokenized_test = dataset["test"].map(tokenize_function, batched=True)

# Rename label column (IMPORTANT)
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_test = tokenized_test.rename_column("label", "labels")

# Remove unnecessary columns
tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_test = tokenized_test.remove_columns(["text"])

# Set format for PyTorch
tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

# 5. Load Model
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

# Add label names (Professional touch)
model.config.id2label = {
    0: "World News",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

model.config.label2id = {
    "World News": 0,
    "Sports": 1,
    "Business": 2,
    "Sci/Tech": 3
}

# 6. Metrics
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy.compute(predictions=predictions, references=labels)["accuracy"]
    f1_score = f1.compute(predictions=predictions, references=labels, average="weighted")["f1"]

    return {
        "accuracy": acc,
        "f1": f1_score
    }

# 7. Training Arguments (FAST + EFFECTIVE)
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    # Reduced for stability
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="no",
    logging_dir="./logs",
    report_to="none"
)

# 8. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics
)

# 9. Train Model
print("Starting training...")
trainer.train()

# 10. Evaluate Model
print("Evaluating model...")
metrics = trainer.evaluate()
print(metrics)

# 11. Save Model
model_path = "./news_topic_model"
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)

# 12. Zip Model
shutil.make_archive("news_topic_model", "zip", model_path)

print("Training complete. Model saved and zipped!")

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will 

Starting training...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

Evaluating model...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.38687536120414734, 'eval_accuracy': 0.882, 'eval_f1': 0.8818603660171759, 'eval_runtime': 191.1585, 'eval_samples_per_second': 2.616, 'eval_steps_per_second': 0.167, 'epoch': 2.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete. Model saved and zipped!


In [7]:
import gradio as gr
import torch
from transformers import BertForSequenceClassification, BertTokenizer, pipeline

model_path = "/content/news_topic_model"
model = BertForSequenceClassification.from_pretrained(model_path)
tokenizer = BertTokenizer.from_pretrained(model_path)

# Use pipeline for multi-class probabilities
classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    top_k=None  # Return all classes
)

# -----------------------------
# Prediction function
# -----------------------------
def predict(text):
    if not text.strip():
        return "Please enter some text."

    # Run classifier
    results = classifier(text)[0]

    # Sort by score descending
    results = sorted(results, key=lambda x: x['score'], reverse=True)

    # Build output string with percentages
    output = "Predictions:\n\n"
    for res in results:
        # Convert label ID to human-readable label
        if res['label'].startswith("LABEL_"):
            label_id = int(res['label'].split("_")[-1])
            label_name = model.config.id2label[label_id]
        else:
            label_name = res['label']
        score = round(res['score'] * 100, 1)  # Display as 99.9% style
        output += f"{label_name}: {score}%\n"

    return output

with gr.Blocks() as demo:

    gr.Markdown("# News Topic Classifier (BERT)")
    gr.Markdown("Classify headlines into World, Sports, Business, or Sci/Tech")

    input_text = gr.Textbox(
        lines=3,
        placeholder="Type your news headline here..."
    )

    with gr.Row():
        predict_btn = gr.Button("Analyze")
        clear_btn = gr.Button("Clear")

    output_text = gr.Textbox(label="Result", interactive=False)

    gr.Examples(
        examples=[
            ["Stock markets surge after tech rally"],
            ["New AI breakthrough in robotics"],
            ["Football team wins championship"],
            ["Global leaders meet for climate summit"]
        ],
        inputs=input_text
    )

    predict_btn.click(fn=predict, inputs=input_text, outputs=output_text)
    input_text.submit(fn=predict, inputs=input_text, outputs=output_text)
    clear_btn.click(lambda: ("", ""), outputs=[input_text, output_text])

if __name__ == "__main__":
    demo.launch(share=True)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://42d6719c02cd756be7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
